In [ ]:
import warnings

warnings.filterwarnings("ignore")

# Patterns to Make the Most of LLMs 

### Structured Output

In [ ]:
from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate


class Joke(BaseModel):
    setup: str = Field(description="The setup to the joke")
    punchline: str = Field(description="The punchline to the joke")


model = ChatOllama(model="llama3.2:3b", temperature=0)
parser = PydanticOutputParser(pydantic_object=Joke)

format_instructions = (
    "Return ONLY a JSON object representing the Joke instance, not a JSON schema. "
    'Use exactly these keys: "setup" (string), "punchline" (string). '
    'Do NOT include keys like "properties", "required", "type", or "title". '
    'No markdown or code fences. Example: {"setup": "…", "punchline": "…"}'
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You output strictly the requested JSON object and nothing else."),
        ("human", "Tell me a joke about {topic}.\n{format_instructions}"),
    ]
).partial(format_instructions=format_instructions)

chain = prompt | model | parser
result = chain.invoke({"topic": "cats"})
print(result)

setup='Why did the cat join a band?' punchline='Because it wanted to be the purr-cussionist.'


**Couple things to notice**
- ***low temprature is usually good fit for structured output, as it reduces the chance the LLM will produce invalid output that does't conform to the schema***
- ***if the LLM produces output that doen't conform to the schema, a validation error is raised. This prevent bad data from silently entring your application***
- ***invoke the model with free-form input. The output returned will automatically match the predefined structured schema***

### Intermediate Output

In [ ]:
from typing import Annotated, TypedDict, List
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, BaseMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages


class State(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]


def create_simple_graph():
    builder = StateGraph(State)

    def llm_node(state: State) -> State:
        reply = model.invoke(state["messages"])
        return {"messages": [reply]}

    builder.add_node("llm", llm_node)
    builder.add_edge(START, "llm")
    builder.add_edge("llm", END)
    return builder.compile()


graph = create_simple_graph()

input = {
    "messages": [
        HumanMessage(
            "how old was the 30th president of the United States when he died?"
        )
    ]
}
for c in graph.stream(input, stream_mode="updates"):
    print(c)

{'llm': {'messages': [AIMessage(content='Calvin Coolidge, the 30th President of the United States, died on January 5, 1933. He was born on July 4, 1872, which means that at the time of his death, he was 60 years old.', additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2025-09-08T11:27:28.250171Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6822090750, 'load_duration': 2574134541, 'prompt_eval_count': 41, 'prompt_eval_duration': 694954667, 'eval_count': 55, 'eval_duration': 3550551000, 'model_name': 'llama3.2:3b'}, id='run--ec54e616-0361-4059-9913-0cbfce3296d3-0', usage_metadata={'input_tokens': 41, 'output_tokens': 55, 'total_tokens': 96})]}}


*Langgraph supports more stream modes*
1. `updates` default mode 
2. `values` yields the current state of the graph very time it changes , that is after each set of nodes finishes executing. This can be useful when the way you display output to your users closely tracks the shape of the graph state.
3. `debug` detailed events every time something happens in your graph, including
    - `checkpoint` events , whenever a new checkpoint of the current state is saved to the database
    - `task` events , emitted whenever a node is about to start running
    - `task_result` events, emitted whenever a node finishes running


### Streaming LLM Output Token by Token

In [ ]:
from typing import Annotated, TypedDict, List
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, BaseMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages


class State(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]


def llm_node(state: State) -> State:
    reply = model.invoke(state["messages"])
    return {"messages": [reply]}


builder = StateGraph(State)
builder.add_node("llm", llm_node)
builder.add_edge(START, "llm")
builder.add_edge("llm", END)
app = builder.compile()


input = {
    "messages": [
        HumanMessage(
            "how old was the 30th president of the United States when he died?"
        )
    ]
}
output = app.astream_events(input, version="v2")
async for event in output:
    if event["event"] == "on_chat_model_stream":
        content = event["data"]["chunk"].content
        if content:
            print(content)

Cal
vin
 Cool
idge
,
 the
 
30
th
 President
 of
 the
 United
 States
,
 died
 on
 January
 
5
,
 
193
3
.
 He
 was
 born
 on
 July
 
4
,
 
187
2
,
 which
 means
 that
 at
 the
 time
 of
 his
 death
,
 he
 was
 
60
 years
 old
.


### Human-in-loop Modalities

#### `interrupt`

In [25]:
import asyncio
from contextlib import aclosing

from typing import TypedDict, Annotated
from langgraph.graph import START, END, StateGraph
from langchain.schema import HumanMessage
from langgraph.graph import StateGraph
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, BaseMessage
from langchain_ollama import ChatOllama


class State(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]


model = ChatOllama(model="llama3.2:3b")


async def main():

    builder = StateGraph(State)

    def llm_call(state: State) -> State:
        result = model.invoke(state["messages"])
        return {"message": result}

    builder.add_node("llm", llm_node)
    builder.add_edge(START, "llm")
    builder.add_edge("llm", END)
    graph = builder.compile(checkpointer=MemorySaver())
    event = asyncio.Event()

    input = {
        "messages": [
            HumanMessage(
                "how old was the 30th president of the United States when he died?"
            )
        ]
    }

    config = {"configurable": {"thread_id": "1"}}

    async with aclosing(graph.astream(input, config)) as stream:
        async for chunk in stream:
            if event.is_set():
                break
            else:
                print(chunk)
    await asyncio.sleep(2)
    event.set()


await main()

{'llm': {'messages': [AIMessage(content='Calvin Coolidge, the 30th President of the United States, died on January 5, 1933, at the age of 61.', additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2025-09-08T12:50:16.825226Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3375178208, 'load_duration': 153720708, 'prompt_eval_count': 41, 'prompt_eval_duration': 486352292, 'eval_count': 33, 'eval_duration': 2731062208, 'model_name': 'llama3.2:3b'}, id='run--d8d04449-cc5b-4092-bf6f-0ec0702f1cb1-0', usage_metadata={'input_tokens': 41, 'output_tokens': 33, 'total_tokens': 74})]}}


#### `Authorize`

In [ ]:
import asyncio
from contextlib import aclosing

from typing import TypedDict, Annotated
from langgraph.graph import START, END, StateGraph
from langchain.schema import HumanMessage
from langgraph.graph import StateGraph
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, BaseMessage
from langchain_ollama import ChatOllama


class State(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]


model = ChatOllama(model="llama3.2:3b")


async def main():

    builder = StateGraph(State)

    def llm_call(state: State) -> State:
        result = model.invoke(state["messages"])
        return {"message": result}

    builder.add_node("llm", llm_node)
    builder.add_edge(START, "llm")
    builder.add_edge("llm", END)
    graph = builder.compile(checkpointer=MemorySaver())
    event = asyncio.Event()

    input = {
        "messages": [
            HumanMessage(
                "how old was the 30th president of the United States when he died?"
            )
        ]
    }

    config = {"configurable": {"thread_id": "1"}}

    output = graph.astream(input, config, interrupt_before=["tools"])
    async for c in output:
        print(c)


await main()

{'llm': {'messages': [AIMessage(content='The 30th President of the United States, Calvin Coolidge, died on January 5, 1933. He was born on July 4, 1872, which made him 60 years old when he died.', additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2025-09-08T13:00:33.091206Z', 'done': True, 'done_reason': 'stop', 'total_duration': 7599403708, 'load_duration': 2510649250, 'prompt_eval_count': 41, 'prompt_eval_duration': 1492593375, 'eval_count': 48, 'eval_duration': 3592424833, 'model_name': 'llama3.2:3b'}, id='run--e115742c-ee90-4882-b3c8-cf6f705a5297-0', usage_metadata={'input_tokens': 41, 'output_tokens': 48, 'total_tokens': 89})]}}


#### *Resume*

In [ ]:
from typing import TypedDict, Annotated, List
from langgraph.graph import START, END, StateGraph
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, BaseMessage
from langchain_ollama import ChatOllama


class State(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]


model = ChatOllama(model="llama3.2:3b")


async def main():
    builder = StateGraph(State)

    def llm_call(state: State) -> State:
        result = model.invoke(state["messages"])
        return {"messages": result}

    builder.add_node("llm", llm_call)
    builder.add_edge(START, "llm")
    builder.add_edge("llm", END)

    graph = builder.compile(checkpointer=MemorySaver())
    config = {"configurable": {"thread_id": "1"}}

    async for c in graph.astream(
        {"messages": [HumanMessage("Test message")]},
        config,
    ):
        print(c)


await main()

{'llm': {'messages': AIMessage(content='Your test message has been received. Is there anything else I can assist you with?', additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2025-09-09T13:09:45.995369Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5187943625, 'load_duration': 2509556959, 'prompt_eval_count': 27, 'prompt_eval_duration': 1579720000, 'eval_count': 18, 'eval_duration': 1096301458, 'model_name': 'llama3.2:3b'}, id='run--43b06a5d-0670-4b29-81c8-ae0f79ab5555-0', usage_metadata={'input_tokens': 27, 'output_tokens': 18, 'total_tokens': 45})}}


#### *Restart*

In [ ]:
# Cell 1: Build the graph once
from typing import TypedDict, Annotated, List
from langgraph.graph import START, END, StateGraph
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, BaseMessage
from langchain_ollama import ChatOllama


class State(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]


model = ChatOllama(model="llama3.2:3b")


def build_graph():
    builder = StateGraph(State)

    def llm_call(state: State) -> State:
        result = model.invoke(state["messages"])
        return {"messages": result}

    builder.add_node("llm", llm_call)
    builder.add_edge(START, "llm")
    builder.add_edge("llm", END)
    return builder.compile(checkpointer=MemorySaver())


graph = build_graph()

In [19]:
# Restart
from langchain_core.messages import HumanMessage

new_input = {"messages": [HumanMessage("New question to start over")]}
new_config = {"configurable": {"thread_id": "run-2"}}

result = await graph.ainvoke(new_input, new_config)
print(result)

{'messages': [HumanMessage(content='New question to start over', additional_kwargs={}, response_metadata={}, id='82ebb97e-a822-4af3-b713-26b7aa748c47'), AIMessage(content="I'd be happy to help with a new question. What's on your mind?", additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2025-09-09T13:16:17.049014Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2584709292, 'load_duration': 94468500, 'prompt_eval_count': 30, 'prompt_eval_duration': 1266949542, 'eval_count': 18, 'eval_duration': 1220885292, 'model_name': 'llama3.2:3b'}, id='run--6f72ae61-41dc-409b-86a4-a616a7160912-0', usage_metadata={'input_tokens': 30, 'output_tokens': 18, 'total_tokens': 48}), HumanMessage(content='New question to start over', additional_kwargs={}, response_metadata={}, id='1f22d5c3-0fb1-4e74-b8d5-968a13e8da6f'), AIMessage(content="Let's start fresh! What would you like to talk about or ask about today?", additional_kwargs={}, response_metadata={'model': 'llama3.2

### *Edit State*

In [ ]:
# gonna use the last agent architecture to use it
config = {"configurable": {"thread_id": "run-1"}}

snap = await graph.aget_state(config)

print(snap.values)
print(snap.next)

{'messages': [HumanMessage(content="please write code to print 2's table in python", additional_kwargs={}, response_metadata={}, id='0ac6a95b-0259-4565-bde5-35d5d30ffc3a')]}
('llm',)


In [ ]:
# Append to messages before resuming

from langchain_core.messages import HumanMessage

await graph.aupdate_state(config, {"messages": [HumanMessage("Checking the messages")]})

InvalidUpdateError: Ambiguous update, specify as_node

In [ ]:
# Replace
from email import message


snap = await graph.aget_state(config)
msgs = snap.values.get("messages", [])

msgs = msgs[:-1]
msgs.append(
    HumanMessage("New final intrustion is write code to print 2's table in python")
)

await graph.aupdate_state(config, {"messages": msgs}, as_node="llm")

{'configurable': {'thread_id': 'run-1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f08d80c-900b-665c-8001-536cdda2743c'}}

In [25]:
# Continue the graph from where it left off
result = await graph.ainvoke(None, config)
print(result)

{'messages': [HumanMessage(content="please write code to print 2's table in python", additional_kwargs={}, response_metadata={}, id='0ac6a95b-0259-4565-bde5-35d5d30ffc3a'), HumanMessage(content="New final intrustion is write code to print 2's table in python", additional_kwargs={}, response_metadata={}, id='d2147a5f-6b19-4718-b124-629b364598d4')]}


#### *Forks*

In [44]:
from typing import TypedDict, Annotated, List
from langgraph.graph import START, END, StateGraph
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage, BaseMessage
from langchain_ollama import ChatOllama

class State(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]

model = ChatOllama(model="llama3.2:3b")

def build_graph():
    builder = StateGraph(State)

    def llm(state: State) -> State:
        result = model.invoke(state["messages"])
        return {"messages": result}

    builder.add_node("llm", llm)
    builder.add_edge(START, "llm")
    builder.add_edge("llm", END)
    return builder.compile(checkpointer=MemorySaver())

graph = build_graph()


In [45]:
def cfg(thread_id: str, checkpoint_id: str | None = None):
    c = {"configurable": {"thread_id": thread_id}}
    if checkpoint_id:
        c["configurable"]["checkpoint_id"] = checkpoint_id
    return c

In [46]:
### First Run from Start
input_1 = {"messages": [HumanMessage("Who was the 30th US President?")]}
config_1 = cfg("run-1")
res1 = await graph.ainvoke(input_1, config_1)
print("Run 1:", res1)

Run 1: {'messages': [HumanMessage(content='Who was the 30th US President?', additional_kwargs={}, response_metadata={}, id='7b109a21-acb6-46e7-bf93-37cc02fe7a76'), AIMessage(content='The 30th President of the United States was Calvin Coolidge.', additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2025-09-09T13:39:21.7379Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4722651208, 'load_duration': 2201256750, 'prompt_eval_count': 34, 'prompt_eval_duration': 1358664917, 'eval_count': 15, 'eval_duration': 1159465458, 'model_name': 'llama3.2:3b'}, id='run--18a418cd-9693-4e22-8c8f-fb43dde4b78f-0', usage_metadata={'input_tokens': 34, 'output_tokens': 15, 'total_tokens': 49})]}


In [47]:

snap = await graph.aget_state(config_1)
print("Keys in state:", snap.values.keys())
print("Next node(s):", snap.next)

Keys in state: dict_keys(['messages'])
Next node(s): ()


In [48]:
await graph.aupdate_state(
    config_1,
    {"messages": [HumanMessage("How old was he when he died?")]},
    as_node="llm"  # attribute update to the node that writes messages
)

{'configurable': {'thread_id': 'run-1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f08d826-515d-6b0c-8002-9ac6efcacf51'}}

In [49]:
res2 = await graph.ainvoke(None, config_1)
print("After append:", res2)

After append: {'messages': [HumanMessage(content='Who was the 30th US President?', additional_kwargs={}, response_metadata={}, id='7b109a21-acb6-46e7-bf93-37cc02fe7a76'), AIMessage(content='The 30th President of the United States was Calvin Coolidge.', additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2025-09-09T13:39:21.7379Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4722651208, 'load_duration': 2201256750, 'prompt_eval_count': 34, 'prompt_eval_duration': 1358664917, 'eval_count': 15, 'eval_duration': 1159465458, 'model_name': 'llama3.2:3b'}, id='run--18a418cd-9693-4e22-8c8f-fb43dde4b78f-0', usage_metadata={'input_tokens': 34, 'output_tokens': 15, 'total_tokens': 49}), HumanMessage(content='How old was he when he died?', additional_kwargs={}, response_metadata={}, id='1468949f-2c7b-4e81-8800-23896d3cd7a0')]}


In [50]:
# Replace entire Message List 
snap2 = await graph.aget_state(config_1)
msgs = snap2.values.get("messages", [])

In [51]:
# Drop last and add new Instructions 
msgs = msgs[:-1]
msgs.append(HumanMessage("Please answer concisely in one sentence."))
await graph.aupdate_state(config_1, {"messages": msgs}, as_node="llm")
res3 = await graph.ainvoke(None, config_1)
print("After replace:", res3)

After replace: {'messages': [HumanMessage(content='Who was the 30th US President?', additional_kwargs={}, response_metadata={}, id='7b109a21-acb6-46e7-bf93-37cc02fe7a76'), AIMessage(content='The 30th President of the United States was Calvin Coolidge.', additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2025-09-09T13:39:21.7379Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4722651208, 'load_duration': 2201256750, 'prompt_eval_count': 34, 'prompt_eval_duration': 1358664917, 'eval_count': 15, 'eval_duration': 1159465458, 'model_name': 'llama3.2:3b'}, id='run--18a418cd-9693-4e22-8c8f-fb43dde4b78f-0', usage_metadata={'input_tokens': 34, 'output_tokens': 15, 'total_tokens': 49}), HumanMessage(content='How old was he when he died?', additional_kwargs={}, response_metadata={}, id='1468949f-2c7b-4e81-8800-23896d3cd7a0'), HumanMessage(content='Please answer concisely in one sentence.', additional_kwargs={}, response_metadata={}, id='3e1de115-ae46-4a3c-84

In [53]:
# Collect snapshots
history = [s async for s in graph.aget_state_history(config_1)]

def get_checkpoint_id(snap):
    # Primary: from config (newer versions)
    cfg = getattr(snap, "config", None)
    if isinstance(cfg, dict):
        return cfg.get("configurable", {}).get("checkpoint_id")
    # Fallback: direct attribute (older versions)
    return getattr(snap, "checkpoint_id", None)

def get_next_nodes(snap):
    return getattr(snap, "next", None)  # some versions may not have this

# Inspect
for i, s in enumerate(history):
    print(i, get_checkpoint_id(s), get_next_nodes(s))

# Revisit last snapshot
cp_id = get_checkpoint_id(history[-1])
revisit_cfg = {"configurable": {"thread_id": "run-1", "checkpoint_id": cp_id}}
alt = await graph.ainvoke(None, revisit_cfg)
print(alt)

0 1f08d826-7ae2-6df6-8003-d986f0f83f72 ()
1 1f08d826-515d-6b0c-8002-9ac6efcacf51 ()
2 1f08d826-46f9-6bde-8001-7beb68d7cf9e ()
3 1f08d826-1888-67d2-8000-ff4487fdd697 ('llm',)
4 1f08d826-1883-6c78-bfff-7f38b61c1692 ('__start__',)
{'messages': [HumanMessage(content='Who was the 30th US President?', additional_kwargs={}, response_metadata={}, id='1781db8d-ef06-4b36-b294-c0fa358cba9a'), AIMessage(content='Calvin Coolidge was the 30th US President. He served from 1923 to 1929.', additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2025-09-09T13:40:18.684576Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2577495333, 'load_duration': 91763833, 'prompt_eval_count': 34, 'prompt_eval_duration': 832556667, 'eval_count': 24, 'eval_duration': 1650868250, 'model_name': 'llama3.2:3b'}, id='run--9b3a0770-ee11-4361-a226-acb98b9de4ec-0', usage_metadata={'input_tokens': 34, 'output_tokens': 24, 'total_tokens': 58})]}
